# LangChain Academy — Module 0 (Google Gemini / AI Studio Edition)

Welcome to **LangChain Academy: Introduction to LangGraph**, configured to use **Google Gemini** models via **Google AI Studio** (`langchain-google-genai`).

---

## 1. Google AI Studio Setup

Your API key and environment are loaded from `.env`.

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain-google-genai langchain-core langchain-community langchain-tavily python-dotenv

## 2. Environment Configuration

The code below loads your `.env` file automatically.

In [2]:
import os, getpass
try:
    from dotenv import load_dotenv
    load_dotenv()  # loads .env
except ImportError:
    pass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("GEMINI_API_KEY")
if os.environ.get("GEMINI_API_KEY") and not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

# LangSmith tracing (optional - enabled only if a real key is present)
langsmith_key = os.environ.get("LANGSMITH_API_KEY", "")
if langsmith_key and not langsmith_key.startswith("your-"):
    os.environ["LANGSMITH_TRACING_V2"] = "true"
    os.environ["LANGSMITH_PROJECT"] = "langchain-academy"
    print("LangSmith tracing enabled!")
else:
    os.environ["LANGSMITH_TRACING"] = "false"

print("Environment configured successfully!")

LangSmith tracing enabled!
Environment configured successfully!


## 3. Initializing Gemini Models

In LangChain, all chat models implement a uniform interface (`invoke`, `stream`, `bind_tools`).
We instantiate `gemini-3.8-flash` (fast, efficient) and `gemini-2.5-pro` (deep reasoning).

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

gemini_flash = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
)

gemini_pro = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0,
)

print("Gemini models initialized successfully!")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Gemini models initialized successfully!


## 4. Invoking the Model

Let's test invoking Gemini with both a structured list of messages and a raw text prompt.

In [6]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="You are an expert AI engineer tutoring on LangGraph and agent orchestration."),
    HumanMessage(content="In 2 concise bullet points, explain why stateful graphs are powerful for agents.")
]

response = gemini_flash.invoke(messages)
# Handle content extraction cleanly
content = response.content
if isinstance(content, list) and len(content) > 0 and isinstance(content[0], dict):
    print(content[0].get("text", content))
else:
    print(content)

* **Native Cyclic Execution & Context Preservation:** Unlike linear pipelines (DAGs), stateful graphs natively support loops and conditional branching, allowing agents to iteratively reason, retry failed tool calls, and self-correct while passing an evolving, shared context between nodes.
* **Durable Persistence & Human-in-the-Loop:** Built-in state checkpointing decouples execution from orchestration, enabling agents to safely pause for human approvals, resume across sessions, and recover from failures or "time-travel" back to earlier states without data loss.


In [7]:
# String inputs are automatically converted to a HumanMessage
response = gemini_flash.invoke("Hello Gemini! Tell me one fun fact about graphs in computer science.")
content = response.content
if isinstance(content, list) and len(content) > 0 and isinstance(content[0], dict):
    print(content[0].get("text", content))
else:
    print(content)

Hello! Here is a fun one:

The famous **Four Color Theorem**—which states that any flat map (a planar graph) can be colored using just four colors without adjacent regions sharing the same color—was the **first major mathematical theorem ever proven by a computer**. 

In 1976, mathematicians Kenneth Appel and Wolfgang Haken couldn't crack it with traditional pencil-and-paper math, so they wrote a program that analyzed **1,936 individual graph configurations** over 1,200 hours of computer time. 

The fun (and chaotic) part? **It caused an existential crisis in the academic world.** Many mathematicians refused to accept the proof at first, arguing that if a human couldn't sit down and verify all the steps by hand, it couldn't truly be considered "math." Today, it is celebrated as a landmark moment that bridged pure mathematics and computer science!


## 5. Streaming Responses

Streaming tokens as they arrive is essential for responsive agent applications.

In [8]:
for chunk in gemini_flash.stream("Explain the concept of Human-in-the-loop in 30 words:"):
    c = chunk.content
    if isinstance(c, list) and len(c) > 0 and isinstance(c[0], dict):
        c = c[0].get("text", "")
    print(c, end="", flush=True)
print()

Human-in-the-loop integrates human judgment into automated artificial intelligence systems. People continuously train, validate, and override algorithmic outputs to combine machine efficiency with human expertise, ensuring safety, accuracy, and ethical accountability.[]


## 6. Tool Calling Preview (Preparation for Module 1)

In LangGraph, agents make decisions and call tools using `.bind_tools()`. Let's verify that tool binding works seamlessly with Gemini.

In [9]:
def add(a: int, b: int) -> int:
    """Adds two integers together.

    Args:
        a: The first integer.
        b: The second integer.
    """
    return a + b

gemini_with_tools = gemini_flash.bind_tools([add])
result = gemini_with_tools.invoke("What is 42 + 58?")
print("Tool calls generated:", result.tool_calls)

Tool calls generated: [{'name': 'add', 'args': {'b': 58, 'a': 42}, 'id': 'call_3602045', 'type': 'tool_call'}]


## 7. Search Tools (Tavily)

Some lessons in later modules use the Tavily Search API for live web grounding.

In [10]:
_set_env("TAVILY_API_KEY")

try:
    from langchain_tavily import TavilySearch
    tavily_search = TavilySearch(max_results=2)
    data = tavily_search.invoke({"query": "What is LangGraph agent orchestration?"})
    search_docs = data.get("results", data)
    print(f"Successfully retrieved {len(search_docs)} search results.")
except Exception as e:
    print("Tavily check skipped or encountered error:", e)

Successfully retrieved 2 search results.
